In [46]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
import torch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

# Фиксация seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

Используется устройство: cuda


In [67]:
import json
import pandas as pd
from pathlib import Path

with open("data/ds_PhilPapersAI.json", "r", encoding="utf-8") as f:
    all_docs = json.load(f)

kb_docs = all_docs[:5]

df_kb = pd.DataFrame([
    {
        "id": f"doc_{i:03d}",
        "text": " ".join(doc["text"]),
        "source": "ds_PhilPapersAI.json"
    }
    for i, doc in enumerate(kb_docs)
])

print(f"Выбрано документов: {len(df_kb)}")
print(f"Средняя длина текста: {df_kb['text'].str.len().mean():.0f} символов")

Выбрано документов: 5
Средняя длина текста: 9750 символов


In [48]:
def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

for idx, row in df_kb.iterrows():
    print(f"\nДокумент {row['id']} (Первые 200 символов текста):")
    print(f"   \"{row['text'][:200]}...\"")
    
    chunks = chunk_text(row['text'], chunk_size=300, overlap=50)
    print(f"Всего чанков: {len(chunks)}")
    print(f"Пример 1-го чанка:\n{chunks[0]}\n")


Документ doc_000 (Первые 200 символов текста):
   "This paper explores some of the factors that make complex systems complex. We first examine the history of complex systems. It was Aristotle's insight that how elements are joined together helps deter..."
Всего чанков: 20
Пример 1-го чанка:
This paper explores some of the factors that make complex systems complex. We first examine the history of complex systems. It was Aristotle's insight that how elements are joined together helps determine the properties of the resulting whole. We find that scientific reductionism does not provide a


Документ doc_001 (Первые 200 символов текста):
   "Douglas Walton's multitudinous contributions to the study of argumentation seldom, if ever, directly engage with argumentation in mathematics. Nonetheless, several of the innovations with which he is ..."
Всего чанков: 18
Пример 1-го чанка:
Douglas Walton's multitudinous contributions to the study of argumentation seldom, if ever, directly engage with 

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

all_chunks = []
for _, row in df_kb.iterrows():
    chunks = chunk_text(row["text"], chunk_size=200, overlap=50)
    for i, ch in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{row['id']}_{i}",
            "doc_id": row["id"],
            "source": row["source"],
            "text": ch
        })

df_chunks = pd.DataFrame(all_chunks)

print("Векторизация чанков...")
chunk_embeddings = embedder.encode(df_chunks["text"].tolist(), show_progress_bar=True, normalize_embeddings=True)

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings.astype(np.float32))

def search_faiss(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q_emb, k)
    results = df_chunks.iloc[ids[0]].copy()
    results["score"] = scores[0]
    return results

test_queries = ["Что такое градиентный спуск?", "Из чего состоит нейросеть?", "Как оптимизировать функцию потерь?"]
for q in test_queries:
    print(f"\nЗапрос: {q}")
    display(search_faiss(q, k=3)[["chunk_id", "doc_id", "text", "score"]])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8450.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Векторизация чанков...


Batches: 100%|██████████| 11/11 [00:00<00:00, 87.95it/s]


🔍 Запрос: Что такое градиентный спуск?


,chunk_id,doc_id,text,score
203,doc_004_19,doc_004,compiled as Angluin (1983). A rearranged versi...,0.209167
191,doc_004_7,doc_004,"rearranged version, with three additions and n...",0.160032
225,doc_004_41,doc_004,ntive as mirth. Some of the items on Angluin's...,0.132650



🔍 Запрос: Из чего состоит нейросеть?


,chunk_id,doc_id,text,score
114,doc_003_2,doc_003,ience. This introduction provides some backgro...,0.199459
240,doc_004_56,doc_004,ponent must make adequate replies to such of t...,0.172658
203,doc_004_19,doc_004,compiled as Angluin (1983). A rearranged versi...,0.172013



🔍 Запрос: Как оптимизировать функцию потерь?


,chunk_id,doc_id,text,score
114,doc_003_2,doc_003,ience. This introduction provides some backgro...,0.218633
186,doc_004_2,doc_004,"ention for several reasons. Firstly, it is wel...",0.175299
9,doc_000_9,doc_000,was known intuitively long before Aristotle. A...,0.164276


In [ ]:
eval_queries = [
    {"query": "How does Aristotle’s concept of 'wholeness' challenge scientific reductionism?", "expected_sources": ["doc_000"]},
    {"query": "What are the three categories of emergent phenomena, and why does the author propose retiring the term?", "expected_sources": ["doc_000"]},
    {"query": "How can Walton’s six dialogue types help contextualize different standards of rigor in mathematical proofs?", "expected_sources": ["doc_001"]},
    {"query": "What role do 'critical questions' play in evaluating defeasible mathematical arguments?", "expected_sources": ["doc_001"]},
    {"query": "Why is equating mathematical proof strictly with formal derivation considered a misrepresentation of actual practice?", "expected_sources": ["doc_002"]},
    {"query": "How can argumentation schemes bridge the gap between rigorous proofs and informal mathematical reasoning?", "expected_sources": ["doc_002"]},
    {"query": "Why do mathematicians frequently rely on informal or non-deductive arguments instead of strict formal derivations?", "expected_sources": ["doc_003"]},
    {"query": "How do cultural and historical contexts influence what counts as a 'valid' mathematical argument?", "expected_sources": ["doc_003"]},
    {"query": "How does Angluin’s list of 'spurious proof methods' reflect real-world anxieties and informal practices in mathematics?", "expected_sources": ["doc_004"]},
    {"query": "Why is humor considered a cognitive mechanism for detecting incongruities in mathematical reasoning?", "expected_sources": ["doc_004"]},
]

def evaluate_retrieval(queries, index, df_chunks, embedder, k=3):
    results = []
    for item in queries:
        res = search_faiss(item["query"], k=k)
        retrieved = res["doc_id"].tolist()
        expected = item["expected_sources"]
        
        hit = int(any(r in expected for r in retrieved))
        relevant_retrieved = len(set(retrieved) & set(expected))
        recall = relevant_retrieved / max(len(set(expected)), 1)
        
        rank_first = next((i+1 for i, r in enumerate(retrieved) if r in expected), None)
        
        results.append({
            "query": item["query"],
            "expected_source": ", ".join(expected),
            "retrieved_sources": ", ".join(retrieved),
            "hit_at_k": hit,
            "recall_at_k": recall,
            "rank_of_first_relevant": rank_first
        })
    return pd.DataFrame(results)

df_eval = evaluate_retrieval(eval_queries, index, df_chunks, embedder, k=3)
display(df_eval)
print(f"Hit@3: {df_eval['hit_at_k'].mean():.2f}")
print(f"Recall@3: {df_eval['recall_at_k'].mean():.2f}")

,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant
0,How does Aristotle’s concept of 'wholeness' ch...,doc_000,"doc_000, doc_000, doc_000",1,1.0,1
1,What are the three categories of emergent phen...,doc_000,"doc_000, doc_000, doc_000",1,1.0,1
2,How can Walton’s six dialogue types help conte...,doc_001,"doc_001, doc_001, doc_001",1,1.0,1
3,What role do 'critical questions' play in eval...,doc_001,"doc_003, doc_001, doc_003",1,1.0,2
4,Why is equating mathematical proof strictly wi...,doc_002,"doc_002, doc_002, doc_002",1,1.0,1
5,How can argumentation schemes bridge the gap b...,doc_002,"doc_002, doc_004, doc_002",1,1.0,1
6,Why do mathematicians frequently rely on infor...,doc_003,"doc_002, doc_003, doc_002",1,1.0,2
7,How do cultural and historical contexts influe...,doc_003,"doc_001, doc_003, doc_002",1,1.0,2
8,How does Angluin’s list of 'spurious proof met...,doc_004,"doc_003, doc_004, doc_003",1,1.0,2
9,Why is humor considered a cognitive mechanism ...,doc_004,"doc_004, doc_004, doc_004",1,1.0,1


Hit@3: 1.00
Recall@3: 1.00


In [ ]:
results_exp = []
for cs in [150, 300]:
    
    temp_chunks = []
    for _, row in df_kb.iterrows():
        chs = chunk_text(row["text"], chunk_size=cs, overlap=50)
        for i, ch in enumerate(chs):
            temp_chunks.append({"chunk_id": f"{row['id']}_{i}", "doc_id": row["id"], "text": ch})
    df_temp = pd.DataFrame(temp_chunks)
    
    embs = embedder.encode(df_temp["text"].tolist(), normalize_embeddings=True)
    idx_temp = faiss.IndexFlatIP(embs.shape[1])
    idx_temp.add(embs.astype(np.float32))
    
    df_temp_eval = evaluate_retrieval(eval_queries, idx_temp, df_temp, embedder, k=3)
    results_exp.append({
        "chunk_size": cs,
        "hit_at_k": df_temp_eval["hit_at_k"].mean(),
        "recall_at_k": df_temp_eval["recall_at_k"].mean()
    })

df_exp = pd.DataFrame(results_exp)
display(df_exp)


,chunk_size,hit_at_k,recall_at_k
0,150,1.0,1.0
1,300,1.0,1.0


In [ ]:
new_docs = [
    {"id": "doc_new_01", "title": "Dropout регуляризация", "text": "Dropout случайно отключает нейроны во время обучения...", "source": "regularization.md"},
    {"id": "doc_new_02", "title": "Learning Rate", "text": "Скорость обучения влияет на сходимость модели...", "source": "training_tips.md"}
]
df_kb_updated = pd.concat([df_kb, pd.DataFrame(new_docs)], ignore_index=True)

all_chunks_upd = []
for _, row in df_kb_updated.iterrows():
    chs = chunk_text(row["text"], chunk_size=200, overlap=50)
    for i, ch in enumerate(chs):
        all_chunks_upd.append({"chunk_id": f"{row['id']}_{i}", "doc_id": row["id"], "source": row["source"], "text": ch})
df_chunks_upd = pd.DataFrame(all_chunks_upd)

embs_upd = embedder.encode(df_chunks_upd["text"].tolist(), normalize_embeddings=True)
index_upd = faiss.IndexFlatIP(embs_upd.shape[1])
index_upd.add(embs_upd.astype(np.float32))

def search_faiss_2(query, index, df_chunks, k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q_emb, k)
    res = df_chunks.iloc[ids[0]].copy()
    res["score"] = scores[0]
    return res

update_queries = ["Как предотвратить переобучение?", "Что такое learning rate?"]
df_update = []
for q in update_queries:
    before = search_faiss(q, k=3)["doc_id"].tolist()
    after_res = search_faiss_2(q, k=3, index=index_upd, df_chunks=df_chunks_upd)
    after = after_res["doc_id"].tolist()
    changed = before != after
    df_update.append({
        "query": q,
        "before_retrieved_sources": ", ".join(before),
        "after_retrieved_sources": ", ".join(after),
        "changed": changed
    })
df_update = pd.DataFrame(df_update)
display(df_update)


,query,before_retrieved_sources,after_retrieved_sources,changed
0,Как предотвратить переобучение?,"doc_004, doc_004, doc_004","doc_new_01, doc_new_02, doc_004",True
1,Что такое learning rate?,"doc_002, doc_004, doc_003","doc_new_01, doc_new_02, doc_002",True


In [ ]:
import os
import json

def mini_rag_pipeline(query, k=3, index=index, df_chunks=df_chunks, embedder=embedder):

    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q_emb, k)
    top_chunks = df_chunks.iloc[ids[0]].copy()
    top_chunks["score"] = scores[0]
    
    context_blocks = []
    for _, row in top_chunks.iterrows():
        block = f"Фрагмент {row['chunk_id']} (источник: {row['source']}, оценка: {row['score']:.3f})\n{row['text']}"
        context_blocks.append(block)
    context = "\n\n---\n\n".join(context_blocks)

    extracted_points = [f"• {row['text'][:250]}..." for _, row in top_chunks.iterrows()]
    answer = (
        f"Контекст:\n{context}\n\n"
        f"Ответ на запрос: \"{query}\"\n"
        f"На основе {k} извлечённых фрагментов система выделяет следующие тезисы:\n"
        + "\n".join(extracted_points) +
        "\n\nОтвет сформирован исключительно на основе предоставленной базы знаний."
    )
    
    sources_str = ", ".join([f"{row['chunk_id']} (score={row['score']:.3f})" for _, row in top_chunks.iterrows()])
    
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": sources_str
    }

test_queries = [
    "What are the main characteristics of complex systems?",
    "How does artificial intelligence relate to human cognition?",
    "What is the role of expert systems in diagnosing plant diseases?"
]

rag_results = []
for q in test_queries:
    print(f"Запрос: {q}")
    res = mini_rag_pipeline(q, k=3)
    rag_results.append(res)
    print(res["answer"][:300] + "...\n" + "="*60)

os.makedirs("artifacts", exist_ok=True)
df_rag = pd.DataFrame(rag_results)
df_rag.to_csv("artifacts/rag_examples.csv", index=False, encoding="utf-8")
print("Mini-RAG завершён. Результаты сохранены в artifacts/rag_examples.csv")

Запрос: What are the main characteristics of complex systems?
Контекст:
Фрагмент doc_000_0 (источник: ds_PhilPapersAI.json, оценка: 0.701)
This paper explores some of the factors that make complex systems complex. We first examine the history of complex systems. It was Aristotle's insight that how elements are joined together helps deter

---

Фрагмент doc_000...
Запрос: How does artificial intelligence relate to human cognition?
Контекст:
Фрагмент doc_004_36 (источник: ds_PhilPapersAI.json, оценка: 0.422)
istics. For example, Marvin Minsky has argued that 'Common sense logic is too unreliable for practical use.' A more sophisticated articulation of this insight characterizes humour as a reward mechanis

---

Фрагмент doc_00...
Запрос: What is the role of expert systems in diagnosing plant diseases?
Контекст:
Фрагмент doc_004_125 (источник: ds_PhilPapersAI.json, оценка: 0.314)
earch through the resultant data set is conclusive. Much mathematical practice concerns the correct applicatio

In [72]:
os.makedirs("artifacts", exist_ok=True)

df_eval[["query", "expected_source", "retrieved_sources", "hit_at_k", "recall_at_k"]].to_csv("artifacts/retrieval_eval.csv", index=False)

df_update.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

